# Simulador interactivo de escenarios — Mundial 2026

Este notebook permite construir escenarios manuales de fase de grupos y recalcular automáticamente:

- tablas de grupo,
- mejores terceros,
- cruces de Round of 32,
- eliminatorias,
- campeón proyectado.

El simulador combina resultados reales ya registrados con marcadores manuales definidos por el usuario.


## 1. Clonar repositorio e instalar dependencias

En Colab, ejecuta esta celda. Si estás corriendo localmente, puedes saltarla.


In [ ]:
import os
from pathlib import Path

REPO_URL = "https://github.com/omargah/WC26-predictor.git"
REPO_DIR = Path("mundial-2026-predictor")

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
print("Directorio actual:", Path.cwd())


In [ ]:
!pip install -q -r requirements.txt
!pip install -q ipywidgets


## 2. Actualizar datos y referencias

Esto descarga resultados nuevos, reconstruye features, predicciones pendientes y referencias para menús.


In [ ]:
!python scripts/update_worldcup_state.py --scope full


## 3. Cargar partidos disponibles


In [ ]:
import pandas as pd
import json
from pathlib import Path
from IPython.display import display, clear_output, Markdown
import ipywidgets as widgets

reference_path = Path("data/reference/available_matches.csv")

if not reference_path.exists():
    raise FileNotFoundError("No existe data/reference/available_matches.csv. Ejecuta primero update_worldcup_state.py")

matches = pd.read_csv(reference_path)
matches["date"] = matches["date"].astype(str)

display(matches[["date", "match", "lambda_home", "lambda_away"]].head(20))
print("Partidos pendientes disponibles:", len(matches))
print("Fechas:", sorted(matches["date"].unique()))


## 4. Constructor interactivo de escenario

Selecciona partido, define marcador y agrégalo al escenario.


In [ ]:
scenario_rows = []

date_dropdown = widgets.Dropdown(
    options=sorted(matches["date"].unique()),
    description="Fecha:",
)

match_dropdown = widgets.Dropdown(
    options=[],
    description="Partido:",
    layout=widgets.Layout(width="650px"),
)

home_goals = widgets.IntSlider(
    value=1,
    min=0,
    max=10,
    step=1,
    description="Local:",
    continuous_update=False,
)

away_goals = widgets.IntSlider(
    value=0,
    min=0,
    max=10,
    step=1,
    description="Visitante:",
    continuous_update=False,
)

scenario_name_box = widgets.Text(
    value="escenario_colab",
    description="Escenario:",
    layout=widgets.Layout(width="450px"),
)

seed_box = widgets.IntText(
    value=42,
    description="Seed:",
)

add_button = widgets.Button(
    description="Agregar marcador",
    button_style="success",
)

clear_button = widgets.Button(
    description="Limpiar escenario",
    button_style="danger",
)

run_button = widgets.Button(
    description="Simular escenario",
    button_style="warning",
)

table_output = widgets.Output()
run_output = widgets.Output()

def refresh_match_options(*args):
    date = date_dropdown.value
    sub = matches[matches["date"] == date].copy()
    options = []
    for idx, row in sub.iterrows():
        label = f"{row['match']}   | λ {row['lambda_home']:.2f}-{row['lambda_away']:.2f}"
        options.append((label, int(idx)))
    match_dropdown.options = options

def show_scenario_table():
    with table_output:
        clear_output()
        if not scenario_rows:
            print("Escenario vacío. Agrega marcadores manuales.")
        else:
            df = pd.DataFrame(scenario_rows)
            display(df)

def add_result(_):
    idx = match_dropdown.value
    row = matches.loc[idx]
    result = {
        "date": row["date"],
        "home_team": row["home_team"],
        "away_team": row["away_team"],
        "home_goals": int(home_goals.value),
        "away_goals": int(away_goals.value),
        "notes": "colab_manual",
    }
    key = (result["date"], result["home_team"], result["away_team"])
    scenario_rows[:] = [r for r in scenario_rows if (r["date"], r["home_team"], r["away_team"]) != key]
    scenario_rows.append(result)
    show_scenario_table()

def clear_scenario(_):
    scenario_rows.clear()
    show_scenario_table()

def run_scenario(_):
    with run_output:
        clear_output()
        if not scenario_rows:
            print("No hay marcadores manuales. Agrega al menos uno.")
            return

        scenario_name = scenario_name_box.value.strip() or "escenario_colab"
        seed = int(seed_box.value)

        input_path = Path("data/scenarios") / f"{scenario_name}_manual_results.csv"
        input_path.parent.mkdir(parents=True, exist_ok=True)

        df = pd.DataFrame(scenario_rows)
        df.to_csv(input_path, index=False, encoding="utf-8")

        print("Archivo de escenario guardado:", input_path)
        display(df)

        !python scripts/simulate_scenario.py --input {str(input_path)} --scenario-name {scenario_name} --seed {seed}

        folder = Path("data/scenarios") / scenario_name

        print("\nCarpeta del escenario:", folder)

        if (folder / "scenario_best_thirds.csv").exists():
            print("\nMejores terceros:")
            display(pd.read_csv(folder / "scenario_best_thirds.csv"))

        if (folder / "scenario_round_of_32.csv").exists():
            print("\nRound of 32:")
            r32 = pd.read_csv(folder / "scenario_round_of_32.csv")
            cols = [c for c in ["match_number", "slot_a", "team_a", "slot_b", "team_b", "same_group_violation"] if c in r32.columns]
            display(r32[cols])

        if (folder / "scenario_full_tournament_summary.json").exists():
            print("\nResumen torneo:")
            summary = json.loads((folder / "scenario_full_tournament_summary.json").read_text(encoding="utf-8"))
            display(summary)

date_dropdown.observe(refresh_match_options, names="value")
add_button.on_click(add_result)
clear_button.on_click(clear_scenario)
run_button.on_click(run_scenario)

refresh_match_options()
show_scenario_table()

display(
    date_dropdown,
    match_dropdown,
    widgets.HBox([home_goals, away_goals]),
    widgets.HBox([scenario_name_box, seed_box]),
    widgets.HBox([add_button, clear_button, run_button]),
    table_output,
    run_output,
)


## 5. Leer archivos generados

Después de simular, puedes leer directamente los archivos de la carpeta `data/scenarios/<nombre_del_escenario>/`.


In [ ]:
from pathlib import Path

scenario_name = scenario_name_box.value.strip() or "escenario_colab"
folder = Path("data/scenarios") / scenario_name

print("Carpeta:", folder)
if folder.exists():
    for p in sorted(folder.iterdir()):
        print("-", p.name)
else:
    print("Todavía no existe. Ejecuta primero el escenario.")
